# Cuantiles en la decision 

In [1]:
import numpy as np
import pandas as pd
pd.set_option('display.width', 180)
pd.set_option('display.max_rows', 120)

In [2]:
CAPITAL_INICIAL = 10_000
COSTO     = 0.001     # 0.1% por operacion
DIAS_ANIO = 252
THETA     = 0.002     # umbral de la senal (0.2%)

pl = pd.read_csv('predicciones_oos.csv', parse_dates=['Fecha'])
cuant = pl[pl['Familia'] == 'cuantil'].copy()
print('Filas cuantil:', len(cuant), '| metodos:', list(cuant['Metodo'].unique()))

Filas cuantil: 27120 | metodos: ['Reg. Lineal Cuantil', 'XGBoost Cuantil', 'Random Forest Cuantil', 'LSTM Cuantil']


In [3]:
def _maquina_estado(compra, venta):
    c = compra.to_numpy(); v = venta.to_numpy()
    pos = np.zeros(len(c), dtype=int); estado = 0
    for i in range(len(c)):
        if estado == 0 and c[i]:
            estado = 1
        elif estado == 1 and v[i]:
            estado = 0
        pos[i] = estado
    return pd.Series(pos, index=compra.index)

def backtest(pos, ret, capital=CAPITAL_INICIAL, costo=COSTO):
    pos = pos.astype(float)
    cambios = pos.diff().abs(); cambios.iloc[0] = abs(pos.iloc[0])
    r = pos * ret - costo * cambios
    r.iloc[-1] = r.iloc[-1] - costo * pos.iloc[-1]
    equity = capital * (1.0 + r).cumprod()
    return r, equity

def metricas_frac(r, equity, pos, ret):
    # entiende pesos en [0,1] (sirve para reglas binarias y de sizing)
    cap_final = equity.iloc[-1]
    ret_total = cap_final / CAPITAL_INICIAL - 1.0
    sigma = r.std(ddof=1)
    sharpe = np.sqrt(DIAS_ANIO) * r.mean() / sigma if sigma > 0 else np.nan
    vol = sigma * np.sqrt(DIAS_ANIO)
    max_dd = (equity / equity.cummax() - 1.0).min()
    invertido = pos > 0
    n_entradas = int(((pos > 0) & (pos.shift(1).fillna(0.0) == 0)).sum())
    turnover = float(pos.diff().abs().sum() + abs(pos.iloc[0]))
    peso_medio = float(pos[invertido].mean()) if invertido.any() else 0.0
    return {'Sharpe': sharpe, 'Retorno Total': ret_total, 'Max Drawdown': max_dd,
            'Volatilidad': vol, '% en mercado': float(invertido.mean()),
            'Peso medio': peso_medio, 'N entradas': n_entradas, 'Turnover': turnover}

def correr(reglas):
    # backtestea un dict {nombre: fn(q10,q50,q90)->posicion} sobre las 4 familias x 3 acciones
    filas = []
    for (ticker, nombre), g in cuant.groupby(['Accion', 'Metodo'], sort=False):
        g = g.sort_values('Fecha'); idx = pd.DatetimeIndex(g['Fecha'].values)
        q10 = pd.Series(g['q10'].values, index=idx)
        q50 = pd.Series(g['q50'].values, index=idx)
        q90 = pd.Series(g['q90'].values, index=idx)
        r_oos = pd.Series(g['ret'].values, index=idx)
        for regla, fn in reglas.items():
            pos = fn(q10, q50, q90)
            r, equity = backtest(pos, r_oos)
            m = metricas_frac(r, equity, pos, r_oos)
            m.update({'Accion': ticker, 'Metodo': nombre, 'Regla': regla})
            filas.append(m)
    return pd.DataFrame(filas).set_index(['Accion', 'Metodo', 'Regla'])

## Definicion de las reglas

In [4]:
# Parametros por defecto
R_RIESGO_DEF = 0.02
P_MIN_DEF    = 0.55

def gate_mediana(q50):
    return _maquina_estado(q50 > THETA, q50 < -THETA).astype(float)

def _prob_supera(theta, q10, q50, q90):
    # CDF lineal por tramos desde los cuantiles 0.10/0.50/0.90 -> P(retorno > theta)
    eps = 1e-12
    pend_baja = 0.40 / (q50 - q10).clip(lower=eps)
    pend_alta = 0.40 / (q90 - q50).clip(lower=eps)
    F = np.where(theta <= q50, 0.10 + (theta - q10) * pend_baja,
                               0.50 + (theta - q50) * pend_alta)
    return pd.Series(1.0 - np.clip(F, 0.0, 1.0), index=q50.index)

# --- reglas (factories para poder barrer parametros) ---
def regla_solo_mediana(q10, q50, q90):
    return gate_mediana(q50)

def regla_conservadora(q10, q50, q90):
    # exige que hasta el escenario pesimista supere el umbral para comprar
    compra = q10 > THETA
    venta  = q90 < -THETA
    return _maquina_estado(compra, venta).astype(float)

def make_veto(r_riesgo):
    def f(q10, q50, q90):
        compra = (q50 > THETA) & (q10 > -r_riesgo)
        venta  = (q50 < -THETA) | (q10 < -r_riesgo)
        return _maquina_estado(compra, venta).astype(float)
    return f

def regla_asimetria(q10, q50, q90):
    upside = q90 - q50; downside = q50 - q10
    compra = (q50 > THETA) & (upside >= downside)
    venta  = (q50 < -THETA) | (upside < downside)
    return _maquina_estado(compra, venta).astype(float)

def make_prob(p_min):
    def f(q10, q50, q90):
        compra = _prob_supera(THETA, q10, q50, q90) >= p_min
        venta  = (1 - _prob_supera(-THETA, q10, q50, q90)) >= p_min
        return _maquina_estado(compra, venta).astype(float)
    return f

REGLAS = {'Solo mediana':           regla_solo_mediana,
          'Conservadora extremos':  regla_conservadora,
          'Veto pesimista':         make_veto(R_RIESGO_DEF),
          'Asimetria r/r':          regla_asimetria,
          'Prob. de ganar':         make_prob(P_MIN_DEF)}
ORDEN = [r for r in REGLAS if r != 'Solo mediana']

In [5]:
maestro = correr(REGLAS)
cols = ['Sharpe', 'Retorno Total', 'Max Drawdown', '% en mercado', 'Peso medio', 'N entradas', 'Turnover']
maestro = maestro[cols]
maestro.to_csv('evidencia_escenarios_maestro.csv', encoding='utf-8')
print('Guardado: evidencia_escenarios_maestro.csv')
maestro.round(3)

Guardado: evidencia_escenarios_maestro.csv


Sharpe  Retorno Total  Max Drawdown  % en mercado  Peso medio  N entradas  Turnover
Accion Metodo                Regla                                                                                                     
NVDA   Reg. Lineal Cuantil   Solo mediana            1.194         41.027        -0.611         0.738         1.0          80     159.0
                             Conservadora extremos   0.803          3.810        -0.369         0.288         1.0           1       1.0
                             Veto pesimista          0.552          1.410        -0.300         0.265         1.0         158     315.0
                             Asimetria r/r           0.495          1.349        -0.309         0.260         1.0         164     328.0
                             Prob. de ganar          0.914         12.386        -0.539         0.688         1.0          31      61.0
       XGBoost Cuantil       Solo mediana            0.769          6.489        -0.582         0.609         1.0         230     459.0
                             Conservadora extremos   1.112         44.233        -0.663         0.940         1.0           2       3.0
                             Veto pesimista          0.106         -0.003        -0.490         0.241         1.0         200     400.0
                             Asimetria r/r           0.167          0.125        -0.466         0.165         1.0         199     397.0
                             Prob. de ganar          0.665          4.024        -0.517         0.588         1.0         115     229.0
       Random Forest Cuantil Solo mediana            0.850          8.584        -0.606         0.569         1.0         218     435.0
                             Conservadora extremos     NaN          0.000         0.000         0.000         0.0           0       0.0
                             Veto pesimista          0.078          0.018        -0.313         0.127         1.0         160     320.0
                             Asimetria r/r           0.333          0.586        -0.547         0.226         1.0         257     513.0
                             Prob. de ganar          0.630          3.405        -0.589         0.559         1.0          98     195.0
       LSTM Cuantil          Solo mediana            1.018         12.462        -0.376         0.517         1.0          54     108.0
                             Conservadora extremos   0.413          1.035        -0.560         0.335         1.0           2       4.0
                             Veto pesimista          0.785          1.619        -0.227         0.160         1.0          43      86.0
                             Asimetria r/r           0.327          0.404        -0.328         0.108         1.0          54     108.0
                             Prob. de ganar          0.856          6.764        -0.383         0.466         1.0          19      38.0
MSFT   Reg. Lineal Cuantil   Solo mediana            1.102          7.068        -0.256         0.773         1.0          50      99.0
                             Conservadora extremos     NaN          0.000         0.000         0.000         0.0           0       0.0
                             Veto pesimista          1.299          7.487        -0.195         0.671         1.0          82     163.0
                             Asimetria r/r           0.515          0.946        -0.258         0.256         1.0         125     250.0
                             Prob. de ganar          0.808          3.472        -0.280         0.732         1.0          18      35.0
       XGBoost Cuantil       Solo mediana            0.909          4.225        -0.259         0.600         1.0         162     323.0
                             Conservadora extremos   0.623          1.843        -0.371         0.575         1.0           1       2.0
                             Veto pesimista          0.492          1.040        -0.319         0.503       

## 2. Delta de Sharpe vs Solo mediana

In [6]:
sh = maestro['Sharpe'].unstack('Regla')
delta = sh[ORDEN].sub(sh['Solo mediana'], axis=0)
delta.to_csv('evidencia_escenarios_delta_sharpe.csv', encoding='utf-8')
print('Guardado: evidencia_escenarios_delta_sharpe.csv')
delta.round(3)

Guardado: evidencia_escenarios_delta_sharpe.csv


Regla                         Conservadora extremos  Veto pesimista  Asimetria r/r  Prob. de ganar
Accion Metodo                                                                                     
GOOGL  LSTM Cuantil                           0.355          -0.440          0.026          -0.064
       Random Forest Cuantil                    NaN           0.156         -0.444           0.045
       Reg. Lineal Cuantil                      NaN           0.082         -0.850          -0.330
       XGBoost Cuantil                       -0.124          -0.585         -0.877          -0.149
MSFT   LSTM Cuantil                          -0.183          -0.172         -0.206          -0.224
       Random Forest Cuantil                 -0.174          -0.585         -0.666          -0.142
       Reg. Lineal Cuantil                      NaN           0.197         -0.587          -0.294
       XGBoost Cuantil                       -0.285          -0.417         -0.780          -0.095
NVDA   LSTM Cuantil                          -0.605          -0.233         -0.691          -0.162
       Random Forest Cuantil                    NaN          -0.772         -0.517          -0.220
       Reg. Lineal Cuantil                   -0.390          -0.642         -0.699          -0.280
       XGBoost Cuantil                        0.343          -0.663         -0.602          -0.105

## 3. Sensibilidad del veto a `R_RIESGO` (1 % / 2 % / 3 %)

Muestra que el veto es monotono: 1 % se pasa de estricto, 3 % converge a la mediana.

In [7]:
reglas_veto = {f'Veto {int(r*100)}%': make_veto(r) for r in [0.01, 0.02, 0.03]}
reglas_veto['Solo mediana'] = regla_solo_mediana
sens_veto = correr(reglas_veto)
sh_v = sens_veto['Sharpe'].unstack('Regla')
delta_veto = sh_v[[f'Veto {p}%' for p in [1, 2, 3]]].sub(sh_v['Solo mediana'], axis=0)
delta_veto.to_csv('evidencia_sensibilidad_veto.csv', encoding='utf-8')
print('Guardado: evidencia_sensibilidad_veto.csv  (Delta Sharpe vs Solo mediana)')
delta_veto.round(3)

Guardado: evidencia_sensibilidad_veto.csv  (Delta Sharpe vs Solo mediana)


Regla                         Veto 1%  Veto 2%  Veto 3%
Accion Metodo                                          
GOOGL  LSTM Cuantil            -0.375   -0.440    0.014
       Random Forest Cuantil   -0.808    0.156    0.136
       Reg. Lineal Cuantil     -1.503    0.082   -0.006
       XGBoost Cuantil         -0.785   -0.585   -0.010
MSFT   LSTM Cuantil            -0.304   -0.172    0.047
       Random Forest Cuantil   -0.220   -0.585   -0.284
       Reg. Lineal Cuantil     -0.851    0.197   -0.060
       XGBoost Cuantil         -0.656   -0.417   -0.047
NVDA   LSTM Cuantil            -0.325   -0.233   -0.310
       Random Forest Cuantil   -0.270   -0.772   -0.398
       Reg. Lineal Cuantil     -1.086   -0.642   -0.395
       XGBoost Cuantil         -0.783   -0.663   -0.309

## 4. Sensibilidad de la probabilidad a `P_MIN` (0.50 / 0.55 / 0.60)

Con `P_MIN = 0.50` la regla ES Solo mediana (`P(ret>theta)>=0.5 <=> Q0.5>=theta`); subir `P_MIN`
solo la vuelve mas exigente -> su techo es la mediana.

In [8]:
reglas_prob = {f'P_MIN={p:.2f}': make_prob(p) for p in [0.50, 0.55, 0.60]}
reglas_prob['Solo mediana'] = regla_solo_mediana
sens_prob = correr(reglas_prob)
sh_p = sens_prob['Sharpe'].unstack('Regla')
delta_prob = sh_p[[f'P_MIN={p:.2f}' for p in [0.50, 0.55, 0.60]]].sub(sh_p['Solo mediana'], axis=0)
delta_prob.to_csv('evidencia_sensibilidad_prob.csv', encoding='utf-8')
print('Guardado: evidencia_sensibilidad_prob.csv  (Delta Sharpe vs Solo mediana)')
delta_prob.round(3)

Guardado: evidencia_sensibilidad_prob.csv  (Delta Sharpe vs Solo mediana)


Regla                         P_MIN=0.50  P_MIN=0.55  P_MIN=0.60
Accion Metodo                                                   
GOOGL  LSTM Cuantil                  0.0      -0.064      -0.076
       Random Forest Cuantil         0.0       0.045      -0.078
       Reg. Lineal Cuantil           0.0      -0.330         NaN
       XGBoost Cuantil               0.0      -0.149      -0.226
MSFT   LSTM Cuantil                  0.0      -0.224      -0.273
       Random Forest Cuantil         0.0      -0.142       0.072
       Reg. Lineal Cuantil           0.0      -0.294      -0.525
       XGBoost Cuantil               0.0      -0.095      -0.032
NVDA   LSTM Cuantil                  0.0      -0.162      -0.252
       Random Forest Cuantil         0.0      -0.220      -0.290
       Reg. Lineal Cuantil           0.0      -0.280      -0.272
       XGBoost Cuantil               0.0      -0.105      -0.180

In [9]:
filas = []
for regla in ORDEN:
    d = delta[regla]
    filas.append({'Regla': regla,
                  'Sharpe medio': sh[regla].mean(),
                  'Delta medio': d.mean(),
                  'Mejoras (de 12)': int((d > 0).sum()),
                  'MaxDD medio': maestro.xs(regla, level='Regla')['Max Drawdown'].mean()})
resumen = pd.DataFrame(filas).set_index('Regla')
resumen.loc['Solo mediana (base)'] = [sh['Solo mediana'].mean(), 0.0, 0,
                                      maestro.xs('Solo mediana', level='Regla')['Max Drawdown'].mean()]
resumen.to_csv('evidencia_resumen.csv', encoding='utf-8')
print('Guardado: evidencia_resumen.csv')
resumen.round(3)

Guardado: evidencia_resumen.csv


,Sharpe medio,Delta medio,Mejoras (de 12),MaxDD medio
Regla,,,,
Conservadora extremos,0.720,-0.133,2.0,-0.254
Veto pesimista,0.512,-0.340,3.0,-0.341
Asimetria r/r,0.277,-0.574,1.0,-0.308
Prob. de ganar,0.683,-0.168,1.0,-0.411
Solo mediana (base),0.851,0.000,0.0,-0.411
